# E-Commerce EDA & Data Preprocessing

## Objectifs

1. **Chargement & Audit** : Importer et valider les données brutes
2. **Nettoyage** : Gérer les valeurs manquantes et aberrantes
3. **Feature Engineering** : Créer des features temporelles et funnel
4. **Analyse Exploratoire** : KPIs et distributions
5. **Export** : Sauvegarder les données nettoyées

Dataset: RetailRocket E-commerce (Kaggle)

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Pandas options
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', None)

# Matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Afficher les versions
print("=" * 60)
print("VERSIONS DES PACKAGES")
print("=" * 60)
print(f"Pandas:     {pd.__version__}")
print(f"NumPy:      {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"Seaborn:    {sns.__version__}")
print("=" * 60)

## 2. Chargement des données

In [ ]:
# Paths
DATA_RAW = Path('../data/raw')
DATA_CLEAN = Path('../data/clean')
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

# Fichiers
events_path = DATA_RAW / 'events.csv'
category_path = DATA_RAW / 'category_tree.csv'
props1_path = DATA_RAW / 'item_properties_part1.csv'
props2_path = DATA_RAW / 'item_properties_part2.csv'

print(f"Chargement depuis: {DATA_RAW}")
print(f"Export vers: {DATA_CLEAN}")

# Chargement
try:
    events = pd.read_csv(events_path, sep=',')
    category_tree = pd.read_csv(category_path, sep=',')
    props1 = pd.read_csv(props1_path, sep=',')
    props2 = pd.read_csv(props2_path, sep=',')
    print("Chargement reussi")
except FileNotFoundError as e:
    print(f"Erreur: {e}")
    raise

print(f"\nshapes:")
print(f"  events: {events.shape}")
print(f"  category_tree: {category_tree.shape}")
print(f"  props1: {props1.shape}")
print(f"  props2: {props2.shape}")

## 3. Audit des données

In [ ]:
def audit_dataframe(df: pd.DataFrame, name: str, head_rows: int = 3) -> None:
    """Audit complet d'un DataFrame."""
    print(f"\n{'='*80}")
    print(f"AUDIT: {name}")
    print(f"{'='*80}")
    
    print(f"\nShape: {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    
    print(f"\nApercu (premieres {head_rows} lignes):")
    display(df.head(head_rows))
    
    print(f"\nTypes & Memoire:")
    info_df = pd.DataFrame({
        'Column': df.columns,
        'Type': df.dtypes.values,
        'Non-Null Count': df.count().values,
        'Null %': (df.isna().sum().values / len(df) * 100).round(2),
    })
    display(info_df)
    
    print(f"\nValeurs manquantes:")
    missing = df.isna().sum().sort_values(ascending=False)
    if missing.sum() > 0:
        missing_pct = (missing / len(df) * 100).round(2)
        display(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0])
    else:
        print("Aucune valeur manquante")
    
    print(f"\nDuplicates: {df.duplicated().sum():,}")

# Audit
audit_dataframe(events, 'EVENTS')
audit_dataframe(category_tree, 'CATEGORY_TREE')
audit_dataframe(props1, 'ITEM_PROPERTIES_PART1')
audit_dataframe(props2, 'ITEM_PROPERTIES_PART2')

## 4. Fusion des propriétés & Nettoyage initial

In [ ]:
# Fusion des propriétés
item_properties = pd.concat([props1, props2], ignore_index=True)
audit_dataframe(item_properties, 'ITEM_PROPERTIES (fused)')

# Déduplicate si nécessaire
item_properties_dedup = item_properties.drop_duplicates()
print(f"\n🔄 Après déduplication: {item_properties_dedup.shape}")

## 5. Prétraitement Events

In [ ]:
# Copie pour traitement
events_clean = events.copy()

print("\n📝 Étapes de nettoyage:\n")

# 1. Conversion du timestamp (RetailRocket: millisecondes)
print("1️⃣ Conversion timestamp (ms → datetime)")
events_clean['timestamp'] = pd.to_datetime(events_clean['timestamp'], unit='ms')
print(f"   Range: {events_clean['timestamp'].min()} à {events_clean['timestamp'].max()}")
print(f"   Durée: {(events_clean['timestamp'].max() - events_clean['timestamp'].min()).days} jours")

# 2. Features temporelles
print("\n2️⃣ Ajout de features temporelles")
events_clean['date'] = events_clean['timestamp'].dt.date
events_clean['hour'] = events_clean['timestamp'].dt.hour
events_clean['day_of_week'] = events_clean['timestamp'].dt.day_name()
events_clean['week'] = events_clean['timestamp'].dt.isocalendar().week
events_clean['month'] = events_clean['timestamp'].dt.month
print("   ✓ date, hour, day_of_week, week, month")

# 3. Indicateurs funnel
print("\n3️⃣ Ajout d'indicateurs funnel")
events_clean['is_view'] = (events_clean['event'] == 'view').astype(int)
events_clean['is_addtocart'] = (events_clean['event'] == 'addtocart').astype(int)
events_clean['is_transaction'] = (events_clean['event'] == 'transaction').astype(int)
print("   ✓ is_view, is_addtocart, is_transaction")

# 4. Nettoyage des valeurs manquantes critiques
print("\n4️⃣ Nettoyage des valeurs manquantes")
print(f"   Avant: {len(events_clean):,} lignes")
events_clean = events_clean.dropna(subset=['itemid', 'visitorid'])
print(f"   Après: {len(events_clean):,} lignes")
print(f"   Supprimées: {events.shape[0] - events_clean.shape[0]:,} lignes")

# 5. Conversion des types
print("\n5️⃣ Conversion des types")
events_clean['itemid'] = events_clean['itemid'].astype(int)
events_clean['visitorid'] = events_clean['visitorid'].astype(int)
if 'categoryid' in events_clean.columns:
    events_clean['categoryid'] = events_clean['categoryid'].astype(int)
print("   ✓ itemid, visitorid, categoryid (int)")

print("\n✅ Nettoyage terminé")
audit_dataframe(events_clean, 'EVENTS_CLEAN')

## 6. Analyse Exploratoire - KPIs

In [ ]:
# Copie pour traitement
events_clean = events.copy()

print("\nEtapes de nettoyage:\n")

# 1. Conversion du timestamp (RetailRocket: millisecondes)
print("1. Conversion timestamp (ms -> datetime)")
events_clean['timestamp'] = pd.to_datetime(events_clean['timestamp'], unit='ms')
print(f"   Range: {events_clean['timestamp'].min()} a {events_clean['timestamp'].max()}")
print(f"   Duree: {(events_clean['timestamp'].max() - events_clean['timestamp'].min()).days} jours")

# 2. Features temporelles
print("\n2. Ajout de features temporelles")
events_clean['date'] = events_clean['timestamp'].dt.date
events_clean['hour'] = events_clean['timestamp'].dt.hour
events_clean['day_of_week'] = events_clean['timestamp'].dt.day_name()
events_clean['week'] = events_clean['timestamp'].dt.isocalendar().week
events_clean['month'] = events_clean['timestamp'].dt.month
print("   date, hour, day_of_week, week, month")

# 3. Indicateurs funnel
print("\n3. Ajout d'indicateurs funnel")
events_clean['is_view'] = (events_clean['event'] == 'view').astype(int)
events_clean['is_addtocart'] = (events_clean['event'] == 'addtocart').astype(int)
events_clean['is_transaction'] = (events_clean['event'] == 'transaction').astype(int)
print("   is_view, is_addtocart, is_transaction")

# 4. Nettoyage des valeurs manquantes critiques
print("\n4. Nettoyage des valeurs manquantes")
print(f"   Avant: {len(events_clean):,} lignes")
events_clean = events_clean.dropna(subset=['itemid', 'visitorid'])
print(f"   Apres: {len(events_clean):,} lignes")
print(f"   Supprimees: {events.shape[0] - events_clean.shape[0]:,} lignes")

# 5. Conversion des types
print("\n5. Conversion des types")
events_clean['itemid'] = events_clean['itemid'].astype(int)
events_clean['visitorid'] = events_clean['visitorid'].astype(int)
if 'categoryid' in events_clean.columns:
    events_clean['categoryid'] = events_clean['categoryid'].astype(int)
print("   itemid, visitorid, categoryid (int)")

print("\nNettoyage termine")
audit_dataframe(events_clean, 'EVENTS_CLEAN')

## 7. Distributions & Visualisations

In [ ]:
# Distribution des événements
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
event_counts = events_clean['event'].value_counts()
axes[0].pie(event_counts.values, labels=event_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Distribution des événements')

# Bar chart
axes[1].bar(event_counts.index, event_counts.values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_ylabel('Nombre')
axes[1].set_title('Nombre d\'événements par type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nDétails:")
print(event_counts)
print(f"\nPourcentages:")
print(event_counts / len(events_clean) * 100)

In [ ]:
# Événements par jour
daily_events = events_clean.groupby('date').size()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_events.index, daily_events.values, linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Date')
ax.set_ylabel('Nombre d\'événements')
ax.set_title('Tendance journalière des événements')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nStatistiques journalières:")
print(f"  Moyenne: {daily_events.mean():.0f}")
print(f"  Médiane: {daily_events.median():.0f}")
print(f"  Min: {daily_events.min():.0f}")
print(f"  Max: {daily_events.max():.0f}")

In [ ]:
# Événements par heure
hourly_events = events_clean.groupby('hour').agg({
    'is_view': 'sum',
    'is_addtocart': 'sum',
    'is_transaction': 'sum'
})

fig, ax = plt.subplots(figsize=(14, 5))
hourly_events.plot(ax=ax, marker='o')
ax.set_xlabel('Heure du jour')
ax.set_ylabel('Nombre d\'événements')
ax.set_title('Distribution des événements par heure')
ax.legend(['Vues', 'Panier', 'Transactions'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Événements par jour de la semaine
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_events = events_clean['day_of_week'].value_counts().reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 5))
dow_events.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Jour de la semaine')
ax.set_ylabel('Nombre d\'événements')
ax.set_title('Distribution des événements par jour de la semaine')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Comportement utilisateurs

In [ ]:
print("\n" + "="*80)
print("KPIs GLOBAUX")
print("="*80)

# Calculs
total_events = len(events_clean)
unique_visitors = events_clean['visitorid'].nunique()
unique_items = events_clean['itemid'].nunique()
unique_categories = events_clean['categoryid'].nunique() if 'categoryid' in events_clean.columns else 0

views = events_clean['is_view'].sum()
addtocart = events_clean['is_addtocart'].sum()
transactions = events_clean['is_transaction'].sum()

# Conversions
conv_view_to_atc = (addtocart / views * 100) if views > 0 else 0
conv_atc_to_trx = (transactions / addtocart * 100) if addtocart > 0 else 0
conv_view_to_trx = (transactions / views * 100) if views > 0 else 0

buyers = events_clean[events_clean['event'] == 'transaction']['visitorid'].nunique()
buyer_rate = (buyers / unique_visitors * 100) if unique_visitors > 0 else 0

# Affichage
kpis = {
    'Total Events': f'{total_events:,}',
    'Unique Visitors': f'{unique_visitors:,}',
    'Unique Items': f'{unique_items:,}',
    'Unique Categories': f'{unique_categories:,}',
    'Period': f'{events_clean["date"].min()} to {events_clean["date"].max()}',
    '': '',
    'Views': f'{int(views):,}',
    'Add to Cart': f'{int(addtocart):,}',
    'Transactions': f'{int(transactions):,}',
    '': '',
    'View > ATC': f'{conv_view_to_atc:.2f}%',
    'ATC > Transaction': f'{conv_atc_to_trx:.2f}%',
    'View > Transaction': f'{conv_view_to_trx:.2f}%',
    'Buyer Rate': f'{buyer_rate:.2f}%',
}

for key, value in kpis.items():
    if key == '':
        print()
    else:
        print(f"{key:.<30} {value}")

## 9. Export des données nettoyées

In [ ]:
# Export events_clean
output_path = DATA_CLEAN / 'events_clean.csv'
events_clean.to_csv(output_path, index=False)
print(f"Exporte: {output_path}")
print(f"  Taille: {output_path.stat().st_size / 1e6:.1f} MB")
print(f"  Lignes: {len(events_clean):,}")
print(f"  Colonnes: {len(events_clean.columns)}")

# Export item_properties
props_output = DATA_CLEAN / 'item_properties_clean.csv'
item_properties_dedup.to_csv(props_output, index=False)
print(f"\nExporte: {props_output}")
print(f"  Taille: {props_output.stat().st_size / 1e6:.1f} MB")

# Export category_tree
cat_output = DATA_CLEAN / 'category_tree_clean.csv'
category_tree.to_csv(cat_output, index=False)
print(f"\nExporte: {cat_output}")

## 10. Résumé & Recommandations

In [ ]:
print("\n" + "="*80)
print("RESUME & RECOMMANDATIONS")
print("="*80)

print("\nDONNEES NETTOYEES:")
print(f"  * {total_events:,} evenements")
print(f"  * {unique_visitors:,} visiteurs uniques")
print(f"  * {unique_items:,} produits uniques")
print(f"  * Periode: {events_clean['date'].min()} a {events_clean['date'].max()}")

print("\nKPIs CLES:")
print(f"  * Taux de conversion (vue->transaction): {conv_view_to_trx:.2f}%")
print(f"  * Taux d'acheteurs: {buyer_rate:.2f}%")
print(f"  * Events par visiteur: {total_events / unique_visitors:.1f}")

print("\nOBSERVATIONS:")
if buyer_rate < 5:
    print("  ALERTE: Faible taux d'acheteurs - Optimiser la conversion")
if conv_view_to_atc > 50:
    print("  ALERTE: Taux vue->panier tres eleve - Verifier les donnees")
if conv_atc_to_trx < 10:
    print("  ALERTE: Panier->transaction tres faible - Friction a l'achat")

print("\nPROCHAINES ETAPES:")
print("  1. Charger events_clean.csv dans le dashboard")
print("  2. Analyser les cohortes et la retention")
print("  3. Simuler des A/B tests")
print("  4. Creer des alertes pour anomalies")
print("  5. Enrichir avec donnees externes (geographie, etc.)")